# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their fields by their @id
record_sets = list(dataset.record_sets.keys())
print(f"Available record sets (@id):\n{record_sets}\n")

# Print out available fields for each record set
for rs_id in record_sets:
    rs = dataset.record_sets[rs_id]
    print(f"Record set: {rs_id}")
    field_ids = [field['@id'] for field in rs['fields']]
    print(f"  Fields (@id): {field_ids}")
    for field in rs['fields']:
        print(f"   - {field['@id']}: {field.get('name', '')}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
# For this dataset, the primary record set is 'dv:dataset', which contains protein and sample data
record_sets = ['dv:dataset']
dataframes = {}

for record_set in record_sets:
    records = list(dataset.records(record_set=record_set))
    dataframes[record_set] = pd.DataFrame(records)

# Explore the extracted DataFrame columns
print(f"Columns in '{record_sets[0]}' DataFrame:")
print(dataframes[record_sets[0]].columns.tolist())

# Show the first few rows
dataframes[record_sets[0]].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Depending on field availability, select a numeric field for analysis.
# Let's inspect some typical quantitative fields matching the dataset's description:
df = dataframes['dv:dataset']
print('First rows:')
display(df.head())

# Try common field names likely found in proteomics/EV studies
possible_numeric_fields = ['coverage_percent', 'peptide_count', 'mw', 'pi', 'normalized_abundance']
available_numeric_fields = [c for c in possible_numeric_fields if c in df.columns]

if len(available_numeric_fields) == 0:
    raise Exception(f"No expected numeric fields found. Available columns: {df.columns.tolist()}")
numeric_field = available_numeric_fields[0]
print(f"Using numeric field for EDA: '{numeric_field}'")

# Filtering: example threshold for numeric value
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean):")
display(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"Normalized '{numeric_field}' for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouping: try to group by a field like 'description' or 'accession'
possible_group_fields = ['description', 'accession', 'sample_id']
group_field = next((c for c in possible_group_fields if c in df.columns), None)
if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped mean '{numeric_field}' by '{group_field}':")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field], bins=30, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If there's another numeric field, let's scatterplot vs the first
other_numeric_fields = [c for c in df.columns if c != numeric_field and pd.api.types.is_numeric_dtype(df[c])]
if len(other_numeric_fields) > 0:
    plt.figure(figsize=(6,6))
    sns.scatterplot(x=df[numeric_field], y=df[other_numeric_fields[0]])
    plt.xlabel(numeric_field)
    plt.ylabel(other_numeric_fields[0])
    plt.title(f'Scatter: {numeric_field} vs {other_numeric_fields[0]}')
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

Through this notebook, we've loaded the FAIR^2 dataset using its Croissant schema, explored its record sets and fields using their `@id`, extracted the main record set `'dv:dataset'` into a DataFrame, and performed basic exploratory analysis and visualization on its numeric fields.

You can extend these analyses to other fields or record sets identified in section 2, and apply more advanced analytical or machine learning techniques as needed.